# Introduction to Information Theory (Exercises and Solutions)


  # Problem 1
  
  The input source to a noisy communication channel is a random variable X over the four symbols a; b; c; d: The output from this channel is a random variable Y over these same four symbols. The joint distribution of these two random variables is as follows:

  ![picture](https://drive.google.com/uc?export=view&id=13ujXXUnxEvuLXts_G1AGY4MHjiqCelpe)

  1. Find out the marginal distribution for X and compute the marginal entropy H(X).
  2. Find out the marginal distribution for Y and compute the marginal entropy H(Y).
  3. What is the joint entropy H(X, Y) of the two random variables?
  4. What is the conditional entropy H(Y|X) and H(X|Y)?
  5. What is the mutual information I(X; Y) between the two random variables?

  Solve questions from 1-5 both in nats and bits units.

In [1]:
import numpy as np
import torch as torch

The Joint probability distribution between X and Y is as follows:

In [120]:
'''
  Input channel, X (a, b, c, d);
  Output channel, Y (a, b, c, d)
'''
# Joint distributions of P(X, Y)

# X corresponds to column and Y corresponds to row
PXY_list = [
            [8, 16, 16, 4],
            [16, 8, 16, 0],
            [32, 32, 16, 0],
            [32, 32, 16, 0]
          ]

PXY = 1.0/np.array(PXY_list)
PXY[PXY == np.inf] = 0
PXY

P_XY = torch.FloatTensor(PXY) #transform to tensor, same probability but now encoded as tensor
P_XY

<ipython-input-120-a4f4e800b0fc>:15: RuntimeWarning: divide by zero encountered in divide
  PXY = 1.0/np.array(PXY_list)


tensor([[0.1250, 0.0625, 0.0625, 0.2500],
        [0.0625, 0.1250, 0.0625, 0.0000],
        [0.0312, 0.0312, 0.0625, 0.0000],
        [0.0312, 0.0312, 0.0625, 0.0000]])

## Defining a function to compute the entropy of a probability distribution

In [121]:
'''
  Input: a probability distribution, P
  Output: entropy H(P) in nats

'''
def compute_entropy(P):
    entropy = (-1.0) * P * P.log()
    H_P = entropy.nansum()
    return H_P

### Defining a function to convert entropy from nat (``e-base`` log) to bits (``2-base`` log)

  Theory:
$$
  log_{2} (H) = log_{2} (e) * log_{e} (H);
$$
$$
  log_{2} (e) = \frac{log_{e} (e)}{log_{e} (2)}
$$

In [122]:
'''
  Input: entropy H(P) in nats unit
  Output: entropy H(P) in bits unit
'''

def nats_to_bits(H_P):
    log_2_e = 1.0/torch.FloatTensor([2.0]).log()
    H_P_bits = log_2_e * H_P

    return H_P_bits.item()

#### **1. Find out the marginal distribution for X**

In [123]:
# The marginal probability distribution P(X)
P_X = P_XY.sum(dim=0) # X corresponds to column, that's why computing it across the row
P_X

print ("The marginal probability distribution of X is {}".format(P_X.tolist()))

The marginal probability distribution of X is [0.25, 0.25, 0.25, 0.25]


 **``Compute the marginal entropy H(X)``**

In [124]:
# The marginal entropy H(X)
H_X = compute_entropy(P_X)
H_X_Bits = nats_to_bits(H_X)


print ("The marginal entropy of the probability distribution of X is {} nats or {} bits".format(H_X, H_X_Bits))

The marginal entropy of the probability distribution of X is 1.3862943649291992 nats or 2.0 bits


#### 2. **Find out the marginal distribution for Y**


In [102]:
# The marginal probability distribution P(Y)
P_Y = P_XY.sum(dim=1) # Y corresponds to row, that's why computing it across the column
P_Y

print ("The marginal probability distribution of Y is {}".format(P_Y.tolist()))

The marginal probability distribution of Y is [0.5, 0.25, 0.125, 0.125]


  **``Compute the marginal entropy H(Y)``**

In [103]:
# The marginal entropy H(Y)
H_Y = compute_entropy(P_Y)
H_Y_Bits = nats_to_bits(H_Y)

print ("The marginal entropy of the probability distribution of Y is {} nats or {} bits".format(H_Y, H_Y_Bits))

The marginal entropy of the probability distribution of Y are 1.2130075693130493 nats or 1.75 bits


#### **3. What is the joint entropy H(X, Y) of the two random variables?**


In [105]:
'''
  Input: P_xy, the joint probability distribution of X and Y
  Output: H(XY), the joint entropy of the probability distribution P_xy
'''
H_XY = compute_entropy(P_XY)
H_XY_Bits = nats_to_bits(H_XY)

print ("The entropy H_XY of the joint probability distribution of P_XY is {} nats or {} bits".format(H_XY, H_XY_Bits))

The marginal entropy of the probability distribution of P_xy are 2.339371919631958 nats or 3.375000238418579 bits


#### **4. What is the conditional entropy H(X|Y) and H(Y|X)?**

  - Computing the conditional probability P(X|Y)

In [149]:
'''
  Input: P_XY, P_Y, the joint and margianl probability distribution of (X,Y) and Y
  Output: P(X|Y), the conditional probability distribution P(X|Y)

    P(X|Y) = P(X,Y)/P(Y)
'''
P_X_given_Y = torch.div(P_XY.T, P_Y).T # since we have to divide P_XY[0] by Y[0]
print ("The conditional probability distribution of P(X|Y) is {}".format(P_X_given_Y.tolist()))

The conditional probability distribution of P(X|Y) is [[0.25, 0.125, 0.125, 0.5], [0.25, 0.5, 0.25, 0.0], [0.25, 0.25, 0.5, 0.0], [0.25, 0.25, 0.5, 0.0]]


- Computing the entropy H(X|Y) of the conditional probability distribution P(X|Y)

In [150]:
'''
  Input: P(X|Y), the conditional probability distribution P(X|Y)
  Output; H(X|Y), the entropy of the conditional probability distribution, P(X|Y)
'''

conditional_entropy = (-1.0) * P_XY * P_X_given_Y.log()
H_X_given_Y = conditional_entropy.nansum()
#compute_entropy(P_X_given_Y) # entropy H(X|Y) of conditional probability, P(X|Y)
H_X_given_Y_Bits = nats_to_bits(H_X_given_Y) # entropy in bits

print ("The entropy of the conditional probability distribution of P(X|Y) is {} nats or {} bits".format(H_X_given_Y, H_X_given_Y_Bits))

The entropy of the conditional probability distribution of P(X|Y) is 1.1263641119003296 nats or 1.6249998807907104 bits


  - Computing the conditional probability P(Y|X)

In [151]:
'''
  Input: P_XY, P_X, the joint and margianl probability distribution of (X,Y) and X
  Output: P(Y|X), the conditional probability distribution P(Y|X)

    P(Y|X) = P(X,Y)/P(X)
'''
P_Y_given_X = P_XY/P_X  # conditional probability, P(Y|X)

print ("The conditional probability distribution of P(Y|X) is {}".format(P_Y_given_X.tolist()))

The conditional probability distribution of P(Y|X) is [[0.5, 0.25, 0.25, 1.0], [0.25, 0.5, 0.25, 0.0], [0.125, 0.125, 0.25, 0.0], [0.125, 0.125, 0.25, 0.0]]


- Computing the entropy H(Y|X) of conditional probability distribution, P(Y|X)

In [152]:
'''
  Input: P(Y|X), the conditional probability distribution P(X|Y)
  Output; H(Y|X), the entropy of the conditional probability distribution, P(Y|X)
'''

conditional_entropy = (-1.0) * P_XY * P_Y_given_X.log()
H_Y_given_X = conditional_entropy.nansum()
#H_Y_given_X = compute_entropy(P_Y_given_X) # entropy H(Y|X) of conditional probability, P(Y|X)
H_Y_given_X_Bits = nats_to_bits(H_Y_given_X) # entropy in bits

print ("The entropy of the conditional probability distribution of P(X|Y) is {} nats or {} bits".format(H_Y_given_X, H_Y_given_X_Bits))

The entropy of the conditional probability distribution of P(X|Y) is 0.9530773758888245 nats or 1.375 bits


#### **5. What is the mutual information I(X; Y) between the two random variables?**

We can use several formulas to compute the mutual information between X and Y
$$
I(X,Y)= H(X) + H(Y) - H(X, Y)
$$

$$
      = H(X) - H(X|Y)
$$

$$
      = H(Y) - H(Y|X)
$$


In [153]:
I_XY = H_X + H_Y - H_XY
I_XY
print ("The mutual information between X and Y, I(X, Y) is {} nats or {} bits".format(I_XY, nats_to_bits(I_XY)))

The mutual information between X and Y, I(X, Y) is 0.259929895401001 nats or 0.3749995529651642 bits


In [154]:
I_XY = H_X - H_X_given_Y
I_XY
print ("The mutual information between X and Y, I(X, Y) is {} nats or {} bits".format(I_XY, nats_to_bits(I_XY)))

The mutual information between X and Y, I(X, Y) is 0.25993025302886963 nats or 0.37500008940696716 bits


In [155]:
I_XY = H_Y - H_Y_given_X
I_XY
print ("The mutual information between X and Y, I(X, Y) is {} nats or {} bits".format(I_XY, nats_to_bits(I_XY)))

The mutual information between X and Y, I(X, Y) is 0.25993019342422485 nats or 0.375 bits
